In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# names.txt is downloaded from https://github.com/karpathy/makemore/blob/master/names.txt
words = open('../data/makemore/names.txt', 'r').read().splitlines()
print(len(words))
print(max(len(w) for w in words))
print(words[:8])

In [ ]:
chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

In [ ]:
block_size = 3  # context length

def build_dataset(words):
    X, Y = [], []
    for w in words:  # Or set the [:5] to look at the examples and check mini-batch overfitting
        # print(w)
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            # print(''.join(itos[i] for i in context), '--->', itos[ix])
            context = context[1:] + [ix]  # Shift the context window

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])      # 80%
Xdev, Ydev = build_dataset(words[n1:n2])  # 10%
Xte, Yte = build_dataset(words[n2:])      # 10%

In [ ]:
def cmp(s, dt, t):
    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | exact {str(ex):5s} | approximate {str(app):5s} | maxdiff {maxdiff}')

In [ ]:
# Initialization
n_embd = 10
n_hidded = 64

g = torch.Generator().manual_seed(2147483647)

# Everything bellow in initialized with randn just for the exercise
# (to not hide the incorrect backprop implementation)
C = torch.randn((vocab_size, n_embd), generator=g)
# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidded), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidded, generator=g) * 0.1  # Use bias just for the exercise despite the BatchNorm
# Layer 2
W2 = torch.randn((n_hidded, vocab_size), generator=g) * 0.1
b2 = torch.randn(vocab_size, generator=g) * 0.1
# BatchNorm parameters
bngain = torch.randn((1, n_hidded), generator=g) * 0.1 + 1.0
bnbias = torch.randn((1, n_hidded), generator=g) * 0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters))
for p in parameters:
    p.requires_grad = True

In [ ]:
# Creating a batch
batch_size = 32
n = batch_size  # An alias for convenience
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix]

In [ ]:
# Forward pass
emb = C[Xb]  # Embed the characters into vectors
embcat = emb.view(emb.shape[0], -1)  # Concatenate the vectors

# Linear layer 1
hprebn = embcat @ W1 + b1  # Hidden layer 1 preactivation
# BatchNorm layer
bnmeani = 1/n * hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = 1/(n-1) * bndiff2.sum(0, keepdim=True)  # (n-1) in denom, because it's a sample, not a population (Bessel's correction)
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias
# Layer 1 activation (non-linearity)
h = torch.tanh(hpreact)
# Linear layer 2
logits = h @ W2 + b2  # Output layer

# Cross entropy loss
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes  # Subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdim=True)
counts_sum_inv = counts_sum**-1  # For bit-identical comparison of grads (PyTorch-like implementation)
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()

# Pytorch backward pass
for p in parameters:
    p.grad = None
for t in [
    logprobs,
    probs,
    counts_sum_inv,
    counts_sum,
    counts,
    norm_logits,
    logit_maxes,
    logits,
    h,
    hpreact,
    bnraw,
    bnvar_inv,
    bnvar,
    bndiff2,
    bndiff,
    bnmeani,
    hprebn,
    embcat,
    emb,
]:
    t.retain_grad()
loss.backward()
loss

# Exercise 1

Backprob through all the variables defined above in the forward pass, then compare the gradients with the PyTorch versions using the `cmp` utility.

In [ ]:
# The derivative is based on:
# loss = -(a + b + c) / 3
# loss = -1/3a - 1/3b - 1/3c
# dloss/da = -1/3
# dloss/db = -1/3
# dloss = -1/n  # The exact formula of the `mean()` function derivative
# All the other derivatives in non-selected columns are equals to zero,
# because all other values are not affecting the outputs - gradients shluld be zero
# The chain rule is absent, because dloss == 1
dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), Yb] = -1.0/n
cmp('logprobs', dlogprobs, logprobs)

# Using just calculus formula
dprobs = (probs**-1) * dlogprobs  # Interpretation: boost the derivatives for low probabilities
cmp('probs', dprobs, probs)

# The derivative here depends on the sahpes:
# counts_sum_inv.shape = (32, 1)
# counts.shape = (32, 27)
# That's why we need to sum the `counts * dprobs` to match the shape
# Note: here the derivative of 2 operations:
# 1: multiplication
# 2: broadcast with copying the second dimension 32 times
dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)

# Using just calculus formula
dcounts_sum = -counts_sum**-2 * dcounts_sum_inv
cmp('counts_sum', dcounts_sum, counts_sum)

# The derivative bellow consists of two parts:
# 1: the branch with `probs` - using the simple calculus formula
# 2: the branch with `counts_sum` - the matrix of 1.0 values, because all the local derivatives are equal (sum)
dcounts = counts_sum_inv * dprobs + torch.ones_like(counts) * dcounts_sum
cmp('counts', dcounts, counts)

dnorm_logits = norm_logits.exp() * dcounts
cmp('norm_logits', dnorm_logits, norm_logits)

# Using just calculus formula + reverse broadcasting
# Note: the dlogit_maxes gradient is too small (~0 or 1e-9)
# This is because tje logit_maxes doesn't influence the softmax value and then the loss
dlogit_maxes = (-dnorm_logits.clone()).sum(1, keepdim=True)
cmp('logit_maxes', dlogit_maxes, logit_maxes)

# The derivative bellow consists of two parts:
# 1: the branch with `norm_logits`: calculus formula (equals 1)
# 2: the branch with `logit_maxes`: the matrix with elements equal to 1 in places where was the max value
dlogits = dnorm_logits.clone() + (F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes)
cmp('logits', dlogits, logits)

# The derivatives bellow are calculated for the matrix multiplication `logits`:
# It can be calculated on the paper from the equasion logits = h @ W2 + b2 ensuring dl/dlogits, which is a chain rule
# Or even simply, remember, that you need to use matrix multiplication
# - try to construct correct multiplications having shapes of dlogits, h, W2 and b2,
# e.g. for dh you need to use dlogits, W2 and one transpose,
# for dW2 you need to use dlogits, h and one transpose :)
# for b2 you need to use dlogits and one sum along some axis :)
dh = dlogits @ W2.T
dW2 = h.T @ dlogits
db2 = dlogits.sum(0, keepdim=True)
cmp('h', dh, h)
cmp('W2', dW2, W2)
cmp('b2', db2, b2)

# Using just calculus formula
dhpreact = (1.0 - h**2) * dh
cmp('hpreact', dhpreact, hpreact)

dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
dbnraw = bngain * dhpreact
dbnbias = dhpreact.sum(0, keepdim=True)
cmp('bngain', dbngain, bngain)
cmp('bnraw', dbnraw, bnraw)
cmp('bnbias', dbnbias, bnbias)

# Reverse broadcasting for derivatives
dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)
# Note: here there are 2 local derivatives, so we apply chain rule 2 times
dbnvar = (-0.5 * (bnvar + 1e-5)**-1.5) * dbnvar_inv
# Here again the pattern, where we need to replicate dimension, which was squashed during forward pass
# We did it earlier workin on `dcounts`
dbndiff2 = 1/(n-1) * torch.ones_like(bndiff2) * dbnvar
# The derivative consists of 2 parts, because of two braches
dbndiff = bnvar_inv * dbnraw + 2 * bndiff * dbndiff2
cmp('bnvar_inv', dbnvar_inv, bnvar_inv)
cmp('bnvar', dbnvar, bnvar)
cmp('bndiff2', dbndiff2, bndiff2)
cmp('bndiff', dbndiff, bndiff)

# Nothing new
dbnmeani = -dbndiff.sum(0, keepdim=True)
dhprebn = dbndiff.clone() + 1/n * torch.ones_like(hprebn) * dbnmeani
cmp('bnmeani', dbnmeani, bnmeani)
cmp('hprebn', dhprebn, hprebn)

# Nothing new
dembcat = dhprebn @ W1.T
dW1 = embcat.T @ dhprebn
db1 = dhprebn.sum(0)
cmp('embcat', dembcat, embcat)
cmp('W1', dW1, W1)
cmp('b1', db1, b1)

# Just re-view the gradients tensor (simple)
demb = dembcat.view(emb.shape)
cmp('emb', demb, emb)

# Re-view the gradients tensor (more difficult)
dC = torch.zeros_like(C)
for k in range(Xb.shape[0]):
    for j in range(Xb.shape[1]):
        # Index of the character out of vocab_size (27)
        ix = Xb[k, j]
        # Put the vector of gradients into the charactr's place
        # We use += to sum up gradients, because the same symbol can appear several times
        dC[ix] += demb[k, j]
cmp('C', dC, C)


# Exercise 2

Backprop through the cross_entropy, all in one go, simplifying the derivatives expression.

In [ ]:
# Forward pass

# Detailed implementation (our own)
# logit_maxes = logits.max(1, keepdim=True).values
# norm_logits = logits - logit_maxes
# counts = norm_logits.exp()
# counts_sum = counts.sum(1, keepdim=True)
# counts_sum_inv = counts_sum**-1
# probs = counts * counts_sum_inv
# logprobs = probs.log()
# loss = -logprobs[range(n), Yb].mean()

# PyTorch implementation
loss_fast = F.cross_entropy(logits, Yb)
print(loss_fast.item(), 'diff', (loss_fast - loss).item())

In [ ]:
# Backward pass

# Detailed implementation (our own)
# dlogprobs = torch.zeros_like(logprobs)
# dlogprobs[range(n), Yb] = -1.0/n
# dprobs = (probs**-1) * dlogprobs
# dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)
# dcounts_sum = -counts_sum**-2 * dcounts_sum_inv
# dcounts = counts_sum_inv * dprobs + torch.ones_like(counts) * dcounts_sum
# dnorm_logits = norm_logits.exp() * dcounts
# dlogit_maxes = (-dnorm_logits.clone()).sum(1, keepdim=True)
# dlogits = dnorm_logits.clone() + (F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes)

# The faster way (the exercise itself)
# For one example from the batch the derivative of the -log(Py) is devided by (Py = e**li / sum(e**lj)):
# 1: i == j: Pi -> dlogits = softmax(logits, 1)
# 2: i != j: Pi - 1 -> dlogits[range(n), Yb] -= 1
# 3: the derivative of the mean => dlogits *= 1/n
dlogits = F.softmax(logits, 1)
dlogits[range(n), Yb] -= 1
dlogits *= 1/n

cmp('logits', dlogits, logits)

In [ ]:
F.softmax(logits, 1)[0]

In [ ]:
dlogits[0] * n  # One of the elements is near -1

In [ ]:
dlogits[0].sum()  # Almost 0

In [ ]:
plt.imshow(dlogits.detach(), cmap='gray')

The cross_entropy loss (or especially the softmax function) works like a push-pull mechanism during the backpropogation: the derivative of the correct answer, where the loss is higher is < 0 (close to -1 in the specific case), which force the training to push parameters so the probability next time will be closer to 1, and all the other elements' derivatives are positive (> 0), which force the training to pull parameters so the peobability next time will be closer to 0.

# Exercise 3

Backprop through the BatchNorm at once, using calculus and simplifying the formula.

In [ ]:
# Forward pass

# Detailed implementation (our own)
# bnmeani = 1/n * hprebn.sum(0, keepdim=True)
# bndiff = hprebn - bnmeani
# bndiff2 = bndiff**2
# bnvar = 1/(n-1) * bndiff2.sum(0, keepdim=True)
# bnvar_inv = (bnvar + 1e-5)**-0.5
# bnraw = bndiff * bnvar_inv
# hpreact = bngain * bnraw + bnbias

# Faster implementation (PyTorch)
hpreact_fast = bngain * (hprebn - hprebn.mean(0, keepdim=True)) / torch.sqrt(hprebn.var(0, keepdim=True, unbiased=True) + 1e-5) + bnbias
print('max diff:', (hpreact_fast - hpreact).abs().max())

In [ ]:
# Backward pass

# Detailed implementation (our own)
# dhpreact = (1.0 - h**2) * dh
# dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
# dbnraw = bngain * dhpreact
# dbnbias = dhpreact.sum(0, keepdim=True)
# dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)
# dbnvar = (-0.5 * (bnvar + 1e-5)**-1.5) * dbnvar_inv
# dbndiff2 = 1/(n-1) * torch.ones_like(bndiff2) * dbnvar
# dbndiff = bnvar_inv * dbnraw + 2 * bndiff * dbndiff2
# dbnmeani = -dbndiff.sum(0, keepdim=True)
# dhprebn = dbndiff.clone() + 1/n * torch.ones_like(hprebn) * dbnmeani

# The faster way (the exercise itself)
# The formula bellow is derived from the calculus formulas and simplified
dhprebn = bngain * bnvar_inv / n * (n * dhpreact - dhpreact.sum(0) - n / (n - 1) * bnraw * (dhpreact * bnraw).sum(0))
cmp('hprebn', dhprebn, hprebn)

# Exercise 4

Put everything together to erase the `loss.backward()`

In [ ]:
# init
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 200 # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647) # for reproducibility
C  = torch.randn((vocab_size, n_embd),            generator=g)
# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden,                        generator=g) * 0.1
# Layer 2
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.1
b2 = torch.randn(vocab_size,                      generator=g) * 0.1
# BatchNorm parameters
bngain = torch.randn((1, n_hidden))*0.1 + 1.0
bnbias = torch.randn((1, n_hidden))*0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True

# same optimization as last time
max_steps = 200000
batch_size = 32
n = batch_size # convenience
lossi = []

# use this context manager for efficiency once your backward pass is written
with torch.no_grad():

  # kick off optimization
  for i in range(max_steps):

    # minibatch construct
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
    Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y

    # forward pass
    emb = C[Xb] # embed the characters into vectors
    embcat = emb.view(emb.shape[0], -1) # concatenate the vectors
    # Linear layer
    hprebn = embcat @ W1 + b1 # hidden layer pre-activation
    # BatchNorm layer
    # -------------------------------------------------------------
    bnmean = hprebn.mean(0, keepdim=True)
    bnvar = hprebn.var(0, keepdim=True, unbiased=True)
    bnvar_inv = (bnvar + 1e-5)**-0.5
    bnraw = (hprebn - bnmean) * bnvar_inv
    hpreact = bngain * bnraw + bnbias
    # -------------------------------------------------------------
    # Non-linearity
    h = torch.tanh(hpreact) # hidden layer
    logits = h @ W2 + b2 # output layer
    loss = F.cross_entropy(logits, Yb) # loss function

    # backward pass
    for p in parameters:
      p.grad = None
    # loss.backward() # use this for correctness comparisons, delete it later!

    # manual backprop! #swole_doge_meme
    # -----------------
    # cross_entropy
    dlogits = F.softmax(logits, 1)
    dlogits[range(n), Yb] -= 1
    dlogits *= 1/n
    # output layer
    dW2 = h.T @ dlogits
    db2 = dlogits.sum(0)
    dh = dlogits @ W2.T
    dhpreact = (1.0 - h**2) * dh
    # BatchNorm
    dhprebn = bngain * bnvar_inv / n * (n * dhpreact - dhpreact.sum(0) - n / (n - 1) * bnraw * (dhpreact * bnraw).sum(0))
    dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
    dbnbias = dhpreact.sum(0, keepdim=True)
    # hidden layer
    dembcat = dhprebn @ W1.T
    dW1 = embcat.T @ dhprebn
    db1 = dhprebn.sum(0)
    # concatenation
    demb = dembcat.view(emb.shape)
    # embedding
    dC = torch.zeros_like(C)
    for k in range(Xb.shape[0]):
        for j in range(Xb.shape[1]):
            ix = Xb[k, j]
            dC[ix] += demb[k, j]
    grads = [dC, dW1, db1, dW2, db2, dbngain, dbnbias]
    # -----------------

    # update
    lr = 0.1 if i < 100000 else 0.01 # step learning rate decay
    for p, grad in zip(parameters, grads):
      # p.data += -lr * p.grad # old way of cheems doge (using PyTorch grad from .backward())
      p.data += -lr * grad # new way of swole doge TODO: enable

    # track stats
    if i % 10000 == 0: # print every once in a while
      print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
    lossi.append(loss.log10().item())

    if i >= 1000: # TODO: delete early breaking when you're ready to train the full net
      break

In [ ]:
# # useful for checking the gradients by comparison with PyTorch (use when `loss.backward()` is present)
# for p, g in zip(parameters, grads):
#   cmp(str(tuple(p.shape)), g, p)

In [ ]:
# calibrate the batch norm at the end of training

with torch.no_grad():
  # pass the training set through
  emb = C[Xtr]
  embcat = emb.view(emb.shape[0], -1)
  hpreact = embcat @ W1 + b1
  # measure the mean/std over the entire training set
  bnmean = hpreact.mean(0, keepdim=True)
  bnvar = hpreact.var(0, keepdim=True, unbiased=True)


In [ ]:
# evaluate train and val loss

@torch.no_grad() # this decorator disables gradient tracking
def split_loss(split):
  x,y = {
    'train': (Xtr, Ytr),
    'val': (Xdev, Ydev),
    'test': (Xte, Yte),
  }[split]
  emb = C[x] # (N, block_size, n_embd)
  embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
  hpreact = embcat @ W1 + b1
  hpreact = bngain * (hpreact - bnmean) * (bnvar + 1e-5)**-0.5 + bnbias
  h = torch.tanh(hpreact) # (N, n_hidden)
  logits = h @ W2 + b2 # (N, vocab_size)
  loss = F.cross_entropy(logits, y)
  print(split, loss.item())

split_loss('train')
split_loss('val')

`loss.backward()`:

```text
train 2.4233431816101074
val 2.421905517578125
```

manual backprop:

```text
train 2.4213039875030518
val 2.419098377227783
```

In [ ]:
# sample from the model
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):

    out = []
    context = [0] * block_size # initialize with all ...
    while True:
      # forward pass
      emb = C[torch.tensor([context])] # (1,block_size,d)
      embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
      hpreact = embcat @ W1 + b1
      hpreact = bngain * (hpreact - bnmean) * (bnvar + 1e-5)**-0.5 + bnbias
      h = torch.tanh(hpreact) # (N, n_hidden)
      logits = h @ W2 + b2 # (N, vocab_size)
      # sample
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break

    print(''.join(itos[i] for i in out))